# RectPack further exploration and summary statistics

More advanced RectPack Exploration, summary statistics

## 1. Define the configuration 

- Start with four algorithms, four sorting rules and two rotation settings: 32 configurations per dataset.
- Keep bin selection fixed initially. Rectpack also supports different bin-selection rules, but adding that factor now would expand the experiment considerably. Sorting is applied in offline mode, which we set explicitly.

In [16]:
import math
import random
import time
from itertools import product

import numpy as np
import pandas as pd

from rectpack import (
    newPacker,
    PackingMode,
    PackingBin,
    MaxRectsBssf,
    MaxRectsBaf,
    SkylineBl,
    GuillotineBafSas,
    SORT_NONE,
    SORT_AREA,
    SORT_SSIDE,
    SORT_LSIDE,
)

ALGORITHMS = {
    "MaxRectsBssf": MaxRectsBssf,
    "MaxRectsBaf": MaxRectsBaf,
    "SkylineBl": SkylineBl,
    "GuillotineBafSas": GuillotineBafSas,
}

SORTS = {
    "input_order": SORT_NONE,
    "area": SORT_AREA,
    "short_side": SORT_SSIDE,
    "long_side": SORT_LSIDE,
}

CONFIGURATIONS = [
    {
        "algorithm": algorithm,
        "sort": sort,
        "rotation": rotation,
    }
    for algorithm, sort, rotation in product(
        ALGORITHMS,
        SORTS,
        [False, True],
    )
]

len(CONFIGURATIONS)

32

Set directory for outputs

In [17]:
from pathlib import Path

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / ".git").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root")

RESULTS_DIR = PROJECT_ROOT / "results" / "rectpack"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Make run_pack() return comparable metrics
- Takes a single configuration.
- Defaults to enough available sheets for the complete-order experiment.
- Records whether the order was completed.
- Calculates an area-based lower bound on sheet count.
- Optionally returns placements, rather than storing every layout in the results table.
- Rectpack returns placements through rect_list() and used sheet dimensions through bin_list()

In [18]:
def run_pack(
    parts,
    algorithm,
    sort,
    rotation,
    sheet_size=(2500, 1250),
    available_sheets=None,
    return_layout=False,
):
    parts = list(parts)

    if not parts:
        raise ValueError("parts must not be empty")

    sheet_w, sheet_h = sheet_size

    if sheet_w <= 0 or sheet_h <= 0:
        raise ValueError("Sheet dimensions must be positive")

    if any(w <= 0 or h <= 0 for _, w, h in parts):
        raise ValueError("Part dimensions must be positive")

    ids = [part_id for part_id, _, _ in parts]
    if len(set(ids)) != len(ids):
        raise ValueError("Part IDs must be unique")

    # With identical sheets, at most one sheet per part is needed,
    # provided each part individually fits.
    if available_sheets is None:
        available_sheets = len(parts)

    if available_sheets < 1:
        raise ValueError("available_sheets must be positive")

    def fits_sheet(w, h):
        return (
            (w <= sheet_w and h <= sheet_h)
            or (
                rotation
                and h <= sheet_w
                and w <= sheet_h
            )
        )

    individually_feasible = all(
        fits_sheet(w, h) for _, w, h in parts
    )

    total_area = sum(w * h for _, w, h in parts)
    sheet_area = sheet_w * sheet_h
    area_lower_bound = math.ceil(total_area / sheet_area)

    start = time.perf_counter()

    packer = newPacker(
        mode=PackingMode.Offline,
        bin_algo=PackingBin.BBF,
        pack_algo=ALGORITHMS[algorithm],
        sort_algo=SORTS[sort],
        rotation=rotation,
    )

    for part_id, w, h in parts:
        packer.add_rect(w, h, rid=part_id)

    packer.add_bin(
        sheet_w,
        sheet_h,
        count=available_sheets,
    )

    packer.pack()

    runtime = time.perf_counter() - start

    placements = packer.rect_list()
    used_sheets = packer.bin_list()

    packed_area = sum(
        w * h for _, _, _, w, h, _ in placements
    )
    used_area = sum(w * h for w, h in used_sheets)

    complete = len(placements) == len(parts)

    result = {
        "algorithm": algorithm,
        "sort": sort,
        "rotation": rotation,
        "bin_selection": "BBF",
        "sheet_width": sheet_w,
        "sheet_height": sheet_h,
        "available_sheets": available_sheets,
        "parts_total": len(parts),
        "parts_packed": len(placements),
        "complete": complete,
        "individually_feasible": individually_feasible,
        "sheets_used": len(used_sheets),
        "requested_area": total_area,
        "packed_area": packed_area,
        "part_fraction": len(placements) / len(parts),
        "area_fraction": packed_area / total_area,
        "utilisation": (
            packed_area / used_area if used_area else 0.0
        ),
        "area_lower_bound": area_lower_bound,
        "sheet_gap_to_area_bound": (
            len(used_sheets) - area_lower_bound
            if complete else np.nan
        ),
        "runtime_seconds": runtime,
    }

    if return_layout:
        result["placements"] = placements
        result["used_sheets"] = used_sheets

    return result

## 3. Generate datasets across sizes and shape profiles
- Do not increase only the number of parts. Include different shape distributions as well.
- The following generator creates three synthetic profiles. Each part fits the sheet in its original orientation, so rotation-disabled runs are not disadvantaged by individually impossible parts.

In [19]:
def make_dataset(
    seed,
    n_parts,
    profile,
    sheet_size=(2500, 1250),
):
    rng = random.Random(seed)
    sheet_w, sheet_h = sheet_size

    profiles = {
        "mixed": [
            (0.08, 0.55, 0.08, 0.55),
        ],
        "strips": [
            (0.45, 0.95, 0.03, 0.15),
            (0.03, 0.15, 0.45, 0.95),
        ],
        "large_and_small": [
            (0.45, 0.80, 0.45, 0.80),
            (0.05, 0.20, 0.05, 0.20),
        ],
    }

    ranges = profiles[profile]
    parts = []

    for i in range(n_parts):
        w_min, w_max, h_min, h_max = rng.choice(ranges)

        w = rng.randint(
            max(1, math.ceil(w_min * sheet_w)),
            math.floor(w_max * sheet_w),
        )
        h = rng.randint(
            max(1, math.ceil(h_min * sheet_h)),
            math.floor(h_max * sheet_h),
        )

        parts.append((f"P{i:05d}", w, h))

    return {
        "dataset_id": f"{profile}_n{n_parts}_seed{seed}",
        "seed": seed,
        "profile": profile,
        "n_parts": n_parts,
        "sheet_size": sheet_size,
        "parts": parts,
    }

**Build a small pilot first**

This gives 30 datasets and 960 configuration runs. After checking runtime and outputs, increase the sizes and number of seeds.

These distributions are test scenarios, not representative manufacturing data.

In [20]:
datasets = [
    make_dataset(
        seed=seed,
        n_parts=n_parts,
        profile=profile,
    )
    for n_parts, profile, seed in product(
        [35, 100],
        ["mixed", "strips", "large_and_small"],
        range(5),
    )
]

## 4. Run the whole experiment
Use the same generated datasets for every configuration. Shuffle execution order to avoid always timing one algorithm first.

In [21]:
def run_benchmark(
    datasets,
    configurations=CONFIGURATIONS,
    repeats=1,
    available_sheets=None,
    execution_seed=123,
):
    rng = random.Random(execution_seed)

    jobs = [
        (dataset, configuration, repeat)
        for dataset in datasets
        for configuration in configurations
        for repeat in range(repeats)
    ]

    rng.shuffle(jobs)

    records = []

    for dataset, configuration, repeat in jobs:
        result = run_pack(
            parts=dataset["parts"],
            sheet_size=dataset["sheet_size"],
            available_sheets=available_sheets,
            **configuration,
        )

        records.append({
            "dataset_id": dataset["dataset_id"],
            "seed": dataset["seed"],
            "profile": dataset["profile"],
            "n_parts": dataset["n_parts"],
            "repeat": repeat,
            **result,
        })

    return pd.DataFrame(records)

In [22]:
results = run_benchmark(
    datasets,
    repeats=1,
)

results.to_csv(
    RESULTS_DIR / "rectpack_benchmark_raw.csv",
    index=False,
)

results.head()

,dataset_id,seed,profile,n_parts,repeat,algorithm,sort,rotation,bin_selection,sheet_width,...,individually_feasible,sheets_used,requested_area,packed_area,part_fraction,area_fraction,utilisation,area_lower_bound,sheet_gap_to_area_bound,runtime_seconds
0,strips_n35_seed2,2,strips,35,0,MaxRectsBaf,long_side,True,BBF,2500,...,True,3,7205179,7205179,1.0,1.0,0.768552,3,0,0.006000
1,large_and_small_n100_seed3,3,large_and_small,100,0,MaxRectsBssf,area,False,BBF,2500,...,True,36,62079781,62079781,1.0,1.0,0.551820,20,16,0.015270
2,mixed_n100_seed3,3,mixed,100,0,SkylineBl,long_side,True,BBF,2500,...,True,13,30411287,30411287,1.0,1.0,0.748586,10,3,0.021217
3,mixed_n100_seed4,4,mixed,100,0,MaxRectsBssf,input_order,True,BBF,2500,...,True,12,30331553,30331553,1.0,1.0,0.808841,10,2,0.003308
4,strips_n35_seed0,0,strips,35,0,MaxRectsBaf,long_side,True,BBF,2500,...,True,3,7474808,7474808,1.0,1.0,0.797313,3,0,0.001163


## 5. Build the summary table


In [23]:
config_cols = ["algorithm", "sort", "rotation"]

dataset_config = (
    results
    .groupby(
        ["dataset_id", "profile", "n_parts"] + config_cols,
        as_index=False,
    )
    .agg(
        complete=("complete", "all"),
        sheets_used=("sheets_used", "first"),
        utilisation=("utilisation", "first"),
        area_fraction=("area_fraction", "first"),
        part_fraction=("part_fraction", "first"),
        sheet_gap_to_area_bound=(
            "sheet_gap_to_area_bound", "first"
        ),
        runtime_seconds=("runtime_seconds", "median"),
    )
)

Check completeness before ranking complete-order results:

In [24]:
completion_summary = (
    dataset_config
    .groupby(config_cols, as_index=False)
    .agg(
        datasets=("dataset_id", "nunique"),
        completion_rate=("complete", "mean"),
    )
)

completion_summary

,algorithm,sort,rotation,datasets,completion_rate
0,GuillotineBafSas,area,False,30,1.0
1,GuillotineBafSas,area,True,30,1.0
2,GuillotineBafSas,input_order,False,30,1.0
3,GuillotineBafSas,input_order,True,30,1.0
4,GuillotineBafSas,long_side,False,30,1.0
5,GuillotineBafSas,long_side,True,30,1.0
6,GuillotineBafSas,short_side,False,30,1.0
7,GuillotineBafSas,short_side,True,30,1.0
8,MaxRectsBaf,area,False,30,1.0
9,MaxRectsBaf,area,True,30,1.0


Stop and inspect any incomplete runs rather than treating them as successful low-sheet solutions:

In [25]:
if not dataset_config["complete"].all():
    raise ValueError(
        "Some runs did not pack the full order. "
        "Inspect them before comparing sheet counts."
    )

No incomplete runs 

Calculate performance relative to the best observed configuration on each dataset, separately for each rotation setting:

In [26]:
comparison_groups = ["dataset_id", "rotation"]

dataset_config["best_observed_sheets"] = (
    dataset_config
    .groupby(comparison_groups)["sheets_used"]
    .transform("min")
)

dataset_config["extra_sheets_vs_best"] = (
    dataset_config["sheets_used"]
    - dataset_config["best_observed_sheets"]
)

dataset_config["relative_excess"] = (
    dataset_config["sheets_used"]
    / dataset_config["best_observed_sheets"]
    - 1
)

dataset_config["tied_best"] = (
    dataset_config["extra_sheets_vs_best"] == 0
)

### **Produce the overall summary:**

Interpretation of the main columns:

| Metric                 | Meaning                                                                                                                      |
| ---------------------- | ---------------------------------------------------------------------------------------------------------------------------- |
| Mean relative excess   | Average proportional sheet overhead versus the best observed method on the same dataset                                      |
| Tied-best rate         | Fraction of datasets where the configuration matched the fewest observed sheets; ties count for every matching configuration |
| Mean gap to area bound | Distance from the area lower bound, not necessarily from the true optimum                                                    |
| Median runtime         | Typical runtime                                                                                                              |
| P95 runtime            | Runtime towards the slow end of the tested workloads                                                                         |

“Best observed” means best among the configurations tested

The area lower bound is

$$
L = \left\lceil \frac{\text{total requested part area}}{\text{sheet area}} \right\rceil
$$

It is only a lower bound: geometric constraints may prevent achieving it. Therefore, sheet_gap_to_area_bound is not a proven optimality gap. However, a complete packing using exactly 
𝐿
L sheets does establish the minimum sheet count for that instance.

In [27]:
summary = (
    dataset_config
    .groupby(config_cols, as_index=False)
    .agg(
        datasets=("dataset_id", "nunique"),
        mean_sheets=("sheets_used", "mean"),
        mean_utilisation=("utilisation", "mean"),
        mean_extra_sheets=("extra_sheets_vs_best", "mean"),
        mean_relative_excess=("relative_excess", "mean"),
        tied_best_rate=("tied_best", "mean"),
        mean_gap_to_area_bound=(
            "sheet_gap_to_area_bound", "mean"
        ),
        median_runtime=("runtime_seconds", "median"),
        p95_runtime=(
            "runtime_seconds",
            lambda x: x.quantile(0.95),
        ),
    )
    .sort_values(
        ["rotation", "mean_relative_excess", "median_runtime"]
    )
)

summary.to_csv(
    RESULTS_DIR / "rectpack_benchmark_summary.csv",
    index=False,
)

formats = {
    "mean_sheets": "{:.2f}",
    "mean_utilisation": "{:.1%}",
    "mean_extra_sheets": "{:.2f}",
    "mean_relative_excess": "{:.1%}",
    "tied_best_rate": "{:.1%}",
    "mean_gap_to_area_bound": "{:.2f}",
    "median_runtime": "{:.4f}",
    "p95_runtime": "{:.4f}",
}

summary_display = summary.copy()

for column, fmt in formats.items():
    summary_display[column] = summary_display[column].map(fmt.format)

display(summary_display)

,algorithm,sort,rotation,datasets,mean_sheets,mean_utilisation,mean_extra_sheets,mean_relative_excess,tied_best_rate,mean_gap_to_area_bound,median_runtime,p95_runtime
20,MaxRectsBssf,long_side,False,30,13.80,68.7%,0.13,1.4%,86.7%,5.03,0.0023,0.0073
12,MaxRectsBaf,long_side,False,30,13.87,68.0%,0.20,2.2%,80.0%,5.10,0.0026,0.0074
4,GuillotineBafSas,long_side,False,30,13.87,67.7%,0.20,3.3%,80.0%,5.10,0.0017,0.0063
16,MaxRectsBssf,area,False,30,13.90,66.7%,0.23,4.6%,80.0%,5.13,0.0029,0.0090
22,MaxRectsBssf,short_side,False,30,13.93,66.5%,0.27,5.0%,73.3%,5.17,0.0024,0.0074
8,MaxRectsBaf,area,False,30,13.90,66.4%,0.23,5.0%,76.7%,5.13,0.0035,0.0097
14,MaxRectsBaf,short_side,False,30,13.93,66.2%,0.27,5.4%,73.3%,5.17,0.0032,0.0076
30,SkylineBl,short_side,False,30,14.33,63.6%,0.67,9.2%,46.7%,5.57,0.0076,0.0347
28,SkylineBl,long_side,False,30,14.53,62.7%,0.87,10.6%,26.7%,5.77,0.0094,0.0346
18,MaxRectsBssf,input_order,False,30,14.57,62.0%,0.90,12.6%,33.3%,5.80,0.0021,0.0064


## 6. Check whether performance changes with size and profile
An overall average can hide a configuration that works well on small mixed datasets but poorly on large strip-heavy ones.

In [28]:
summary_by_workload = (
    dataset_config
    .groupby(
        ["profile", "n_parts"] + config_cols,
        as_index=False,
    )
    .agg(
        datasets=("dataset_id", "nunique"),
        mean_relative_excess=("relative_excess", "mean"),
        tied_best_rate=("tied_best", "mean"),
        median_runtime=("runtime_seconds", "median"),
    )
    .sort_values(
        [
            "profile",
            "n_parts",
            "rotation",
            "mean_relative_excess",
            "median_runtime",
        ]
    )
)

For example, inspect the larger strip datasets:

In [29]:
summary_by_workload.query(
    "profile == 'strips' and n_parts == 100"
)

,profile,n_parts,algorithm,sort,rotation,datasets,mean_relative_excess,tied_best_rate,median_runtime
180,strips,100,MaxRectsBssf,long_side,False,5,0.025000,0.8,0.003304
164,strips,100,GuillotineBafSas,long_side,False,5,0.028571,0.8,0.002126
182,strips,100,MaxRectsBssf,short_side,False,5,0.053571,0.6,0.003215
172,strips,100,MaxRectsBaf,long_side,False,5,0.053571,0.6,0.003491
176,strips,100,MaxRectsBssf,area,False,5,0.057143,0.8,0.003557
174,strips,100,MaxRectsBaf,short_side,False,5,0.078571,0.4,0.003760
168,strips,100,MaxRectsBaf,area,False,5,0.078571,0.4,0.003788
188,strips,100,SkylineBl,long_side,False,5,0.103571,0.2,0.010415
190,strips,100,SkylineBl,short_side,False,5,0.207937,0.0,0.013479
178,strips,100,MaxRectsBssf,input_order,False,5,0.210714,0.2,0.003315
